In [1]:
# Stage 5 — PPE Representation Study
# 5.3 — Probability Pollinator Matrix (B′)
#
# Goal: Replace binary pollinator existence matrix P with a continuous
# probability matrix PMp, where PMp[species, bin, week] = normalized
# weekly observation count from GBIF (insects + birds combined).
# PCA PMp to 15D → Vp_prob. Pair with existing binary Vf.
# Feature vector: [Vf (15D), Vp_prob (15D), N (1D)] = 31D
# Compare against A2, A', B, and A3.
#
# PMp construction:
#   - Source: GBIF insects (gbif_0007192) + birds (gbif_0007204), combined
#   - Spatial bins: 0.5° × 0.5°, snapped to common_bins grid
#   - Temporal bins: doy → week index via (doy - 1) // 7, clipped to [0, 51]
#   - Normalization: per (species, bin), weekly counts / total counts
#   - MIN_OBS = 5: bins with fewer than 5 total observations are zero-padded

import numpy as np
import pandas as pd
import pickle
from collections import defaultdict
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = "/scratch/ariana.l/Stage 4 Link Prediction Model/"
STAGE5_BASE = "/scratch/ariana.l/Stage 5 PPE Representation Study/"
GBIF_INSECT = "/scratch/ariana.l/Plant Pollinator Initial Analysis/gbif_0007192_observations_v2.csv"
GBIF_BIRD   = "/scratch/ariana.l/Plant Pollinator Initial Analysis/gbif_0007204_observations_v2.csv"

# Load common bins
print("Loading existence matrices...")
F = pd.read_csv(BASE + "stage4_F_existence_phenofield.csv", index_col=0)
P = pd.read_csv(BASE + "stage4_P_existence_gbif_combined.csv", index_col=0)

common_bins = sorted(set(F.columns) & set(P.columns))
bin_to_idx = {b: i for i, b in enumerate(common_bins)}
n_bins = len(common_bins)  # 3160
n_weeks = 52
print(f"Common bins: {n_bins}")

def snap(x):
    return round(round(x * 2) / 2, 1)

def fmt(x):
    return f"{x:.1f}"

Loading existence matrices...
Common bins: 3160


In [3]:
# Cell 2 — Load and combine GBIF insect + bird observations

print("Loading GBIF insects...")
ins = pd.read_csv(GBIF_INSECT, usecols=['pollinator_species', 'lat', 'lon', 'doy'],
                  low_memory=False)
print(f"  Insects: {len(ins):,} rows")

print("Loading GBIF birds...")
bird = pd.read_csv(GBIF_BIRD, usecols=['pollinator_species', 'lat', 'lon', 'doy'],
                   low_memory=False)
print(f"  Birds: {len(bird):,} rows")

gbif = pd.concat([ins, bird], ignore_index=True)
print(f"Combined: {len(gbif):,} rows, {gbif['pollinator_species'].nunique():,} species")

# Convert doy to week index
gbif['week'] = ((gbif['doy'] - 1) // 7).clip(0, 51).astype(int)

# Snap coordinates to common bin grid
gbif['bin_key'] = gbif['lat'].map(snap).map(fmt) + '_' + gbif['lon'].map(snap).map(fmt)
gbif = gbif[gbif['bin_key'].isin(bin_to_idx)]
gbif['bin_idx'] = gbif['bin_key'].map(bin_to_idx)
print(f"After filtering to common bins: {len(gbif):,} rows")

Loading GBIF insects...
  Insects: 5,053,625 rows
Loading GBIF birds...
  Birds: 542,898,437 rows
Combined: 547,952,062 rows, 4,515 species
After filtering to common bins: 504,347,182 rows


In [4]:
# Cell 3 — Build PMp (n_species × n_bins × n_weeks)

print("Aggregating weekly counts per (species, bin)...")
counts = gbif.groupby(['pollinator_species', 'bin_idx', 'week']).size().reset_index(name='count')

# Total observations per (species, bin) for MIN_OBS filter and normalization
totals = counts.groupby(['pollinator_species', 'bin_idx'])['count'].sum().reset_index(name='total')
counts = counts.merge(totals, on=['pollinator_species', 'bin_idx'])

# Apply MIN_OBS filter
MIN_OBS = 5
counts = counts[counts['total'] >= MIN_OBS]
print(f"Rows after MIN_OBS={MIN_OBS} filter: {len(counts):,}")

# Normalize: weekly count / total per (species, bin)
counts['norm'] = counts['count'] / counts['total']

# Get ordered pollinator species list
pol_species = sorted(gbif['pollinator_species'].dropna().unique())
pol_to_idx = {s: i for i, s in enumerate(pol_species)}
n_species = len(pol_species)
print(f"Pollinator species: {n_species}")

# Allocate PMp
print(f"Allocating PMp: ({n_species}, {n_bins}, {n_weeks})...")
PMp = np.zeros((n_species, n_bins, n_weeks), dtype=np.float32)
print(f"Memory usage: {PMp.nbytes / 1e9:.2f} GB")

# Fill PMp
print("Filling PMp...")
for row in counts.itertuples(index=False):
    sp_i = pol_to_idx.get(row.pollinator_species)
    if sp_i is not None:
        PMp[sp_i, row.bin_idx, row.week] = row.norm

print(f"PMp shape: {PMp.shape}")
print(f"Non-zero entries: {np.count_nonzero(PMp):,} / {PMp.size:,} ({100*np.count_nonzero(PMp)/PMp.size:.1f}%)")

Aggregating weekly counts per (species, bin)...
Rows after MIN_OBS=5 filter: 7,822,389
Pollinator species: 4425
Allocating PMp: (4425, 3160, 52)...
Memory usage: 2.91 GB
Filling PMp...
PMp shape: (4425, 3160, 52)
Non-zero entries: 7,822,389 / 727,116,000 (1.1%)


In [5]:
# Cell 4 — Fit PCA to 15D → Vp_prob

PMp_flat = PMp.reshape(n_species, n_bins * n_weeks)
print(f"Flattened shape: {PMp_flat.shape}")

print("Fitting PCA (randomized, 15 components)...")
pca_pmp = PCA(n_components=15, svd_solver='randomized', random_state=42)
Vp_prob = pca_pmp.fit_transform(PMp_flat)
print(f"Vp_prob shape: {Vp_prob.shape}")
print(f"Variance explained per component: {pca_pmp.explained_variance_ratio_.round(3)}")
print(f"Total variance explained: {pca_pmp.explained_variance_ratio_.sum():.3f}")

Flattened shape: (4425, 164320)
Fitting PCA (randomized, 15 components)...
Vp_prob shape: (4425, 15)
Variance explained per component: [0.197 0.059 0.035 0.024 0.016 0.013 0.011 0.01  0.009 0.008 0.008 0.007
 0.007 0.007 0.006]
Total variance explained: 0.415


In [6]:
# Cell 5 — Save Vp_prob and free memory

Vp_prob_df = pd.DataFrame(Vp_prob, index=pol_species, columns=[f'PC{i+1}' for i in range(15)])
Vp_prob_df.to_csv(STAGE5_BASE + "stage5_Vp_prob.csv")
print(f"Saved Vp_prob: {Vp_prob_df.shape}")

with open(STAGE5_BASE + "stage5_pca_pmp.pkl", "wb") as f:
    pickle.dump(pca_pmp, f)
print("Saved PCA object")

del PMp, PMp_flat
import gc; gc.collect()
print("Freed PMp from memory")

Saved Vp_prob: (4425, 15)
Saved PCA object
Freed PMp from memory


In [7]:
# Cell 6 — Reconstruct training pairs and assemble 31D feature vectors

print("Loading assets...")
Vf_df = pd.read_csv(BASE + "stage4_Vf_phenofield.csv", index_col=0)
globi = pd.read_csv(BASE + "stage4_globi_conus_broad.csv")

F_common = F[common_bins]
P_common = P[common_bins]

# Positive pairs
pos_pairs = globi[['sourceTaxonName', 'targetTaxonName']].drop_duplicates()
pos_pairs.columns = ['pollinator', 'plant']
pos_pairs = pos_pairs[
    pos_pairs['plant'].isin(Vf_df.index) &
    pos_pairs['plant'].isin(F.index) &
    pos_pairs['pollinator'].isin(Vp_prob_df.index) &
    pos_pairs['pollinator'].isin(P.index)
]
pos_pairs['label'] = 1
print(f"Positive pairs: {len(pos_pairs)}")

# Negative pairs
pos_set = set(zip(pos_pairs['pollinator'], pos_pairs['plant']))
all_plants = list(set(Vf_df.index) & set(F.index))
all_pols = list(set(Vp_prob_df.index) & set(P.index))

np.random.seed(42)
neg_pairs = []
while len(neg_pairs) < len(pos_pairs) * 3:
    pol = np.random.choice(all_pols)
    plant = np.random.choice(all_plants)
    if (pol, plant) not in pos_set:
        neg_pairs.append((pol, plant, 0))

neg_pairs = pd.DataFrame(neg_pairs, columns=['pollinator', 'plant', 'label'])
print(f"Negative pairs: {len(neg_pairs)}")

pairs = pd.concat([pos_pairs, neg_pairs], ignore_index=True)

# Assemble 31D feature vectors
def build_features(row):
    vf = Vf_df.loc[row.plant].values              # 15D — binary plant embedding
    vp = Vp_prob_df.loc[row.pollinator].values    # 15D — spatiotemporal pollinator embedding
    N  = float(np.dot(F_common.loc[row.plant].values, P_common.loc[row.pollinator].values))
    return np.concatenate([vf, vp, [N]])           # 31D

print("Assembling feature matrix...")
X = np.vstack([build_features(row) for row in pairs.itertuples()])
y = pairs['label'].values
print(f"X shape: {X.shape}, positive rate: {y.mean():.3f}")

Loading assets...
Positive pairs: 3148
Negative pairs: 9444
Assembling feature matrix...
X shape: (12592, 31), positive rate: 0.250


In [8]:
# Cell 7 — Train/test split and logistic regression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

clf_bprime = LogisticRegression(max_iter=1000, random_state=42)
clf_bprime.fit(X_train, y_train)

y_prob = clf_bprime.predict_proba(X_test)[:, 1]
roc = roc_auc_score(y_test, y_prob)
pr = average_precision_score(y_test, y_prob)

print(f"\nB' (31D, PMp) Results:")
print(f"  ROC-AUC: {roc:.3f}")
print(f"  PR-AUC:  {pr:.3f}")
print(f"\nFor reference:")
print(f"  A2 (31D, binary):           ROC-AUC 0.931, PR-AUC 0.842")
print(f"  A' (35D, V_delta appended): ROC-AUC 0.937, PR-AUC 0.855")
print(f"  B  (31D, PMf):              ROC-AUC 0.938, PR-AUC 0.856")
print(f"  A3 (32D, scalar delta):     ROC-AUC 0.950, PR-AUC 0.868")

Train: (10073, 31), Test: (2519, 31)

B' (31D, PMp) Results:
  ROC-AUC: 0.924
  PR-AUC:  0.828

For reference:
  A2 (31D, binary):           ROC-AUC 0.931, PR-AUC 0.842
  A' (35D, V_delta appended): ROC-AUC 0.937, PR-AUC 0.855
  B  (31D, PMf):              ROC-AUC 0.938, PR-AUC 0.856
  A3 (32D, scalar delta):     ROC-AUC 0.950, PR-AUC 0.868
